# Step 5: 컨텍스트 관리 — 시스템 프롬프트 조립

## 학습 목표

- Claude Code가 **정적/동적 섹션을 분리**하여 프롬프트 캐시를 최적화하는 방법을 이해합니다
- **memoize 패턴**(`@lru_cache`)으로 반복 계산을 피하는 방법을 배웁니다
- **ToolUseContext**가 도구 실행 환경을 캡슐화하는 패턴을 이해합니다

> **왜 시스템 프롬프트 조립이 중요한가?**
> LLM에 보내는 시스템 프롬프트는 에이전트의 "두뇌 설정"입니다.
> OS 정보, 프로젝트 규칙(CLAUDE.md), 사용 가능한 도구 목록 등을 포함하며,
> 이를 효율적으로 조립하고 캐싱하면 비용과 지연 시간을 크게 줄일 수 있습니다.

---

## Claude Code 소스 분석

### 5-1. 시스템 프롬프트의 구조

Claude Code의 시스템 프롬프트는 여러 소스에서 조립됩니다:

```
시스템 프롬프트 = [정적 부분] + [동적 부분]

┌─────────────────────────────────────────────────┐
│  정적 부분 (캐시 가능 - SYSTEM_PROMPT_STATIC)     │
│  ┌─────────────────────────────────────────────┐ │
│  │  MAIN_PROMPT (역할, 행동 지침)                │ │
│  │  TOOL_USE_RULES (도구 사용 규칙)              │ │
│  │  도구 설명 (잘 변하지 않음)                    │ │
│  └─────────────────────────────────────────────┘ │
│ ─ ─ ─ SYSTEM_PROMPT_DYNAMIC_BOUNDARY ─ ─ ─ ─ ─  │
│  동적 부분 (매 요청마다 변함)                      │
│  ┌─────────────────────────────────────────────┐ │
│  │  시스템 컨텍스트 (OS, CWD, git branch)        │ │
│  │  사용자 컨텍스트 (CLAUDE.md)                  │ │
│  │  현재 시간, 세션별 정보                        │ │
│  └─────────────────────────────────────────────┘ │
└─────────────────────────────────────────────────┘
```

**왜 정적/동적을 분리하나?**

Anthropic API의 **프롬프트 캐싱**은 시스템 프롬프트의 앞부분이 이전 요청과 같으면 캐시에서 재사용합니다. 정적 부분을 앞에, 동적 부분을 뒤에 배치하면 캐시 히트율이 높아져 비용이 90%까지 절감됩니다.

### 5-2. getSystemPrompt() — 프롬프트 조립 (prompts.ts)

```typescript
// src/constants/prompts.ts — getSystemPrompt()

export function getSystemPrompt(options: SystemPromptOptions): string {
  const sections: string[] = [];

  // ── 정적 섹션 (캐시 가능) ──
  sections.push(MAIN_PROMPT);              // 역할/행동 지침
  sections.push(TOOL_USE_RULES);           // 도구 사용 규칙

  // 도구별 시스템 프롬프트 (도구가 변하지 않는 한 캐시 가능)
  for (const tool of options.tools) {
    if (tool.systemPromptFragment) {
      sections.push(tool.systemPromptFragment);
    }
  }

  // ── 동적 경계 ──
  // 이 지점 이후의 내용은 매 요청마다 달라질 수 있음
  sections.push(SYSTEM_PROMPT_DYNAMIC_BOUNDARY);

  // ── 동적 섹션 ──
  sections.push(getSystemContext());        // OS, CWD, git
  sections.push(getUserContext());          // CLAUDE.md

  return sections.join("\n\n");
}
```

### 5-3. memoize 패턴 (context.ts:36-180)

```typescript
// src/context.ts — getUserContext(), getSystemContext() with memoize

// memoize: 같은 인자로 호출하면 캐시된 결과를 반환
// 세션 시작 시 한 번만 계산하고 이후 재사용

const getSystemContext = memoize(() => {
  // OS 정보, CWD, git branch 등을 수집
  // 이 정보는 세션 동안 변하지 않으므로 캐시해도 안전
  return {
    os: process.platform,
    cwd: process.cwd(),
    shell: process.env.SHELL,
    gitBranch: getGitBranch(),
    // ...
  };
});

const getUserContext = memoize(() => {
  // CLAUDE.md 파일들을 찾아 내용을 합침
  // 프로젝트 루트, 홈 디렉토리, 부모 디렉토리 순으로 탐색
  return loadClaudeMdFiles();
});
```

### 5-4. systemPromptSection() vs DANGEROUS_uncachedSystemPromptSection()

```typescript
// src/constants/systemPromptSections.ts

// 캐시 가능한 섹션 — 정적 경계 앞에 배치
function systemPromptSection(title: string, content: string): string {
  return `## ${title}\n${content}`;
}

// 캐시 불가능한 섹션 — 동적 경계 뒤에 배치
// "DANGEROUS_" 접두사는 캐시를 무효화할 수 있음을 경고
function DANGEROUS_uncachedSystemPromptSection(title, content): string {
  return `## ${title}\n${content}`;
}
// 이름에 DANGEROUS_를 붙여서 개발자가 "이 함수를 쓰면 캐시가 깨진다"는 것을
// 명확히 인지하게 합니다. 코드 리뷰에서도 즉시 눈에 띕니다.
```

### 5-5. ToolUseContext — 도구 실행 환경 캡슐화 (Tool.ts:158-300)

```typescript
// src/Tool.ts:158-300 — ToolUseContext

// 도구가 실행될 때 필요한 모든 환경 정보를 하나의 객체로 전달
type ToolUseContext = {
  cwd: string;              // 현재 작업 디렉토리
  readFileTimestamps: Map;  // 파일 읽기 타임스탬프 (편집 충돌 감지)
  abortController: AbortController; // 도구 실행 취소
  options: {
    dangerouslySkipPermissions: boolean;
    // ...
  };
};

// 모든 도구는 call(input, context: ToolUseContext) 형태로 호출됨
// 도구가 필요한 환경 정보를 직접 수집하지 않고, context에서 받음
// → 의존성 주입(DI) 패턴
```

---

## Python으로 구현하기

`mini_claude/context.py`에 구현된 `ContextManager`를 확인합니다.

### 핵심 구성 요소

1. **STATIC_PROMPT / TOOL_USE_RULES** — 고정 텍스트 (역할, 지침)
2. **get_system_context()** — OS/CWD/git 정보 (`@lru_cache`로 캐시)
3. **get_user_context()** — CLAUDE.md 로드 (`@lru_cache`로 캐시)
4. **ToolUseContext** — 도구 실행 환경 캡슐화
5. **get_system_prompt()** — 모든 섹션을 조립

In [ ]:
import sys, os
sys.path.insert(0, ".")

from mini_claude.context import (
    ContextManager, ToolUseContext,
    STATIC_PROMPT, TOOL_USE_RULES,
)

# --- 정적 프롬프트 확인 ---
print("=== 정적 프롬프트 (STATIC_PROMPT) ===")
print(STATIC_PROMPT)
print()
print("=== 도구 사용 규칙 (TOOL_USE_RULES) ===")
print(TOOL_USE_RULES)
print()

# --- ToolUseContext: 도구 실행 환경 캡슐화 ---
print("=== ToolUseContext (의존성 주입 객체) ===")
ctx = ToolUseContext()
print(f"  cwd:        {ctx.cwd}")
print(f"  is_git_repo: {ctx.is_git_repo}")
print(f"  git_branch:  {ctx.git_branch}")
print(f"  env 변수 수: {len(ctx.env)}개")
print()
print("ToolUseContext는 도구가 직접 os.getcwd()를 호출하는 대신")
print("context.cwd를 사용하게 하여, 테스트와 모킹을 쉽게 합니다.")

### @lru_cache로 memoize: 시스템/사용자 컨텍스트

`get_system_context()`와 `get_user_context()`는 `@lru_cache(maxsize=1)`로 데코레이트되어 있습니다.
세션 시작 시 한 번만 계산하고, 이후 호출에서는 캐시된 결과를 즉시 반환합니다.

In [ ]:
import time

# ContextManager 생성 (현재 디렉토리 기준)
cm = ContextManager(cwd=os.getcwd())

# --- get_system_context(): OS/CWD/git 정보 ---
print("=== 시스템 컨텍스트 (@lru_cache 적용) ===")
print(cm.get_system_context())
print()

# --- @lru_cache 효과 확인 ---
# 첫 호출: 실제 계산 (git 명령 실행 등)
# 두 번째 호출: 캐시에서 즉시 반환

# 캐시 초기화 후 측정
cm.get_system_context.cache_clear()

start = time.perf_counter()
result1 = cm.get_system_context()
first_call = (time.perf_counter() - start) * 1000

start = time.perf_counter()
result2 = cm.get_system_context()
second_call = (time.perf_counter() - start) * 1000

print(f"첫 번째 호출: {first_call:.3f}ms (실제 계산)")
print(f"두 번째 호출: {second_call:.3f}ms (캐시 히트)")
print(f"캐시 효과: {first_call / max(second_call, 0.001):.0f}x 빠름")
print(f"결과 동일: {result1 is result2}")  # 같은 객체 (캐시)
print()

# --- get_user_context(): CLAUDE.md 로드 ---
print("=== 사용자 컨텍스트 (CLAUDE.md) ===")
user_ctx = cm.get_user_context()
if user_ctx:
    print(user_ctx[:500])
else:
    print("(CLAUDE.md 파일이 없습니다 — 빈 문자열 반환)")
print()

# --- build_tool_context(): 도구 실행 환경 생성 ---
print("=== ToolUseContext 생성 ===")
tool_ctx = cm.build_tool_context()
print(f"  cwd:        {tool_ctx.cwd}")
print(f"  is_git_repo: {tool_ctx.is_git_repo}")
print(f"  git_branch:  {tool_ctx.git_branch}")

---

### 실습: 전체 시스템 프롬프트 조립

`get_system_prompt()`를 호출하여 모든 섹션이 어떻게 합쳐지는지 확인합니다.
빌트인 도구를 포함시켜 도구 설명 섹션도 생성합니다.

In [ ]:
from mini_claude.tools.bash_tool import BashTool
from mini_claude.tools.file_read_tool import FileReadTool
from mini_claude.tools.file_write_tool import FileWriteTool

# 도구 인스턴스 생성
tools = [BashTool(), FileReadTool(), FileWriteTool()]

# 전체 시스템 프롬프트 조립
cm2 = ContextManager(cwd=os.getcwd())
system_prompt = cm2.get_system_prompt(tools=tools)

# 섹션별로 분리하여 출력
print("=== 전체 시스템 프롬프트 ===")
print(f"총 길이: {len(system_prompt)} 문자")
print()

# 섹션 경계를 시각적으로 표시
sections = system_prompt.split("\n\n")
for i, section in enumerate(sections):
    # 각 섹션의 첫 줄만 표시
    first_line = section.split("\n")[0][:80]
    line_count = section.count("\n") + 1
    print(f"  [{i+1}] {first_line}... ({line_count}줄)")

print()
print("=" * 60)
print("Claude Code의 SYSTEM_PROMPT_DYNAMIC_BOUNDARY에 해당하는 지점:")
print()

# 정적 vs 동적 경계 시뮬레이션
static_parts = [STATIC_PROMPT, TOOL_USE_RULES]
static_len = sum(len(p) for p in static_parts)
dynamic_len = len(system_prompt) - static_len

print(f"  정적 부분: ~{static_len} 문자 (캐시 가능 - 비용 절감)")
print(f"  동적 부분: ~{dynamic_len} 문자 (매 요청마다 변함)")
print(f"  캐시 가능 비율: {static_len / len(system_prompt) * 100:.0f}%")

In [ ]:
# 전체 시스템 프롬프트를 실제로 출력하여 내용을 확인합니다
print("=== 전체 시스템 프롬프트 (전문) ===")
print("=" * 60)
print(system_prompt)
print("=" * 60)

---

## 핵심 설계 원칙 정리

### 1. 정적/동적 경계로 캐시 최적화
시스템 프롬프트를 정적(캐시 가능) 부분과 동적(매번 변하는) 부분으로 나누면, Anthropic API의 프롬프트 캐싱을 최대한 활용할 수 있습니다. Claude Code는 `SYSTEM_PROMPT_DYNAMIC_BOUNDARY`로 이 경계를 명시합니다.

### 2. memoize로 반복 계산 제거
`get_system_context()`와 `get_user_context()`는 세션 동안 결과가 변하지 않으므로 `@lru_cache`로 캐시합니다. OS 정보 수집, git 명령 실행, 파일 읽기 등의 비용을 한 번만 지불합니다.

### 3. 의존성 주입으로 테스트 용이성 확보
`ToolUseContext`는 도구가 필요한 환경 정보(CWD, 환경변수, git 상태)를 하나의 객체로 캡슐화합니다. 도구가 직접 `os.getcwd()`를 호출하는 대신 `context.cwd`를 사용하므로, 테스트에서 mock context를 주입할 수 있습니다.

### 4. DANGEROUS_ 접두사로 의도를 명시
`DANGEROUS_uncachedSystemPromptSection()`은 캐시를 무효화할 수 있는 함수입니다. 함수 이름에 `DANGEROUS_`를 붙여 개발자가 의도적으로만 사용하게 합니다. 이런 네이밍 규칙은 코드 리뷰에서 즉시 주의를 끕니다.

---

## 다음 Step 미리보기

**Step 6: Hooks**에서는 에이전트의 동작을 런타임에 커스터마이징하는 방법을 배웁니다:
- ~20개의 **라이프사이클 포인트**(PreToolUse, PostToolUse 등)에서 커스텀 로직 실행
- **셸 명령어 기반 훅**으로 언어 독립적인 확장
- 훅이 **권한 시스템과 연결**되어 도구 실행을 차단하거나 수정하는 패턴